In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd

NOTEBOOK_WORKING_DIRECTORY = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        candidate_directory
        for candidate_directory in (
            NOTEBOOK_WORKING_DIRECTORY,
            *NOTEBOOK_WORKING_DIRECTORY.parents,
        )
        if (
            (candidate_directory / "README.md").is_file()
            and (candidate_directory / "reports" / "tables").is_dir()
            and (candidate_directory / "models" / "metrics").is_dir()
        )
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository root containing README.md, "
        "reports/tables/, and models/metrics/."
    )

os.chdir(PROJECT_ROOT)

TABLE_DIRECTORY = PROJECT_ROOT / "reports" / "tables"
REPORT_DIRECTORY = PROJECT_ROOT / "reports" / "evaluation"

REPORT_DIRECTORY.mkdir(parents=True, exist_ok=True)

with (
    PROJECT_ROOT
    / "models"
    / "metrics"
    / "phase5_final_holdout_evaluation.json"
).open(encoding="utf-8") as file:
    final_holdout_evaluation = json.load(file)

with (
    PROJECT_ROOT
    / "models"
    / "metrics"
    / "calibration_comparison_validation.json"
).open(encoding="utf-8") as file:
    calibration_comparison = json.load(file)

phase7b_policy_summary = pd.read_csv(
    TABLE_DIRECTORY / "phase7b_policy_error_summary.csv"
).iloc[0].to_dict()

phase7a_global_importance = pd.read_csv(
    TABLE_DIRECTORY / "phase7a_global_shap_importance.csv"
)

phase7a_stability_summary = pd.read_csv(
    TABLE_DIRECTORY / "phase7a_shap_stability_summary.csv"
)

phase7d_business_value = pd.read_csv(
    TABLE_DIRECTORY / "phase7d_business_value_summary.csv"
).iloc[0].to_dict()

holdout_result = final_holdout_evaluation["holdout_result"]
champion_model = final_holdout_evaluation["champion_model"]

sigmoid_result = calibration_comparison["results"]["sigmoid"]
uncalibrated_result = calibration_comparison["results"]["uncalibrated"]
isotonic_result = calibration_comparison["results"]["isotonic"]

ablation_definitions = [
    {
        "experiment": "Isolation Forest anomaly-score feature",
        "path": TABLE_DIRECTORY / "phase6_anomaly_policy_comparison.csv",
        "candidate_capture_column": "anomaly_captured_fraud",
        "candidate_value_column": "anomaly_net_expected_value",
    },
    {
        "experiment": "Time-safe graph features",
        "path": TABLE_DIRECTORY / "phase6_graph_policy_comparison.csv",
        "candidate_capture_column": "graph_captured_fraud",
        "candidate_value_column": "graph_net_expected_value",
    },
    {
        "experiment": "Compact LSTM sequence model",
        "path": TABLE_DIRECTORY / "phase6_sequence_policy_comparison.csv",
        "candidate_capture_column": "lstm_captured_fraud",
        "candidate_value_column": "lstm_net_expected_value",
    },
]

ablation_rows = []

for definition in ablation_definitions:
    if not definition["path"].is_file():
        continue

    comparison_table = pd.read_csv(definition["path"])

    capacity_1000_rows = comparison_table.loc[
        comparison_table["review_capacity"] == 1000
    ]

    if capacity_1000_rows.empty:
        continue

    comparison_row = capacity_1000_rows.iloc[0]

    baseline_capture = int(
        comparison_row["baseline_captured_fraud"]
    )
    candidate_capture = int(
        comparison_row[definition["candidate_capture_column"]]
    )
    baseline_value = float(
        comparison_row["baseline_net_expected_value"]
    )
    candidate_value = float(
        comparison_row[definition["candidate_value_column"]]
    )

    ablation_rows.append(
        {
            "experiment": definition["experiment"],
            "review_capacity": 1000,
            "baseline_captured_fraud": baseline_capture,
            "candidate_captured_fraud": candidate_capture,
            "captured_fraud_difference": (
                candidate_capture - baseline_capture
            ),
            "baseline_net_expected_value": baseline_value,
            "candidate_net_expected_value": candidate_value,
            "net_expected_value_difference": (
                candidate_value - baseline_value
            ),
            "selection_decision": (
                "Rejected as champion replacement"
                if candidate_value <= baseline_value
                else "Requires documented comparison review"
            ),
        }
    )

ablation_summary = pd.DataFrame(ablation_rows)

if not ablation_summary.empty:
    ablation_summary.to_csv(
        TABLE_DIRECTORY / "phase7e_advanced_ablation_summary.csv",
        index=False,
    )

top_global_features = phase7a_global_importance.head(10).copy()

stability_mean_minimum = float(
    phase7a_stability_summary["mean_jaccard_similarity"].min()
)

stability_mean_maximum = float(
    phase7a_stability_summary["mean_jaccard_similarity"].max()
)


def markdown_table(
    table: pd.DataFrame,
    columns: list[str],
    decimal_columns: list[str] | None = None,
) -> str:
    """Create a Markdown table without requiring optional packages."""
    decimal_columns = decimal_columns or []

    display_table = table.loc[:, columns].copy()

    for column in decimal_columns:
        if column in display_table.columns:
            display_table[column] = display_table[column].map(
                lambda value: (
                    f"{float(value):,.6f}"
                    if pd.notna(value)
                    else ""
                )
            )

    display_table = display_table.fillna("")

    for column in display_table.columns:
        display_table[column] = (
            display_table[column]
            .astype(str)
            .str.replace("|", "\\|", regex=False)
        )

    header = "| " + " | ".join(display_table.columns) + " |"
    separator = "| " + " | ".join(
        ["---"] * len(display_table.columns)
    ) + " |"

    rows = [
        "| " + " | ".join(row) + " |"
        for row in display_table.astype(str).values.tolist()
    ]

    return "\n".join([header, separator, *rows])


ablation_section = (
    markdown_table(
        ablation_summary,
        columns=[
            "experiment",
            "review_capacity",
            "baseline_captured_fraud",
            "candidate_captured_fraud",
            "captured_fraud_difference",
            "baseline_net_expected_value",
            "candidate_net_expected_value",
            "net_expected_value_difference",
            "selection_decision",
        ],
        decimal_columns=[
            "baseline_net_expected_value",
            "candidate_net_expected_value",
            "net_expected_value_difference",
        ],
    )
    if not ablation_summary.empty
    else "No Phase 6 ablation comparison tables were available."
)

final_model_selection_report = f"""# Final Model Selection and Evaluation Report

## Executive Summary

This portfolio project evaluates fraud prioritisation using leakage-safe
chronological validation, point-in-time behavioural features, calibrated
probabilities, explicit cost assumptions, and constrained investigation capacity.

The selected champion is the Phase 5 XGBoost model, version
`{champion_model["model_version"]}`, with sigmoid / Platt probability calibration
and a fixed top-`{int(holdout_result["review_capacity"]):,}` review-capacity policy.
The final chronological holdout evaluation captured
`{int(holdout_result["captured_fraud_count"]):,}` of
`{int(holdout_result["total_fraud_count"]):,}` fraud-labelled transactions
(`{float(holdout_result["fraud_capture_rate"]):.6f}` capture rate).

This is a public IEEE-CIS benchmark evaluation. The results do not represent
real financial-institution performance, realised fraud savings, or a live
production decision system.

## Decision Objective

The system ranks transactions by fraud risk and supports constrained
approve/review/escalation reasoning under explicit benchmark assumptions.

The central question is not simply whether a transaction is predicted as fraud.
It is:

> Which transactions should receive limited investigation capacity, how confident
> is the model, what model features contributed to prioritisation, and what is the
> expected decision trade-off under documented assumptions?

## Selected Champion

| Item | Selected value |
| --- | --- |
| Model family | XGBoost classifier |
| Model version | `{champion_model["model_version"]}` |
| MLflow run ID | `{champion_model["mlflow_run_id"]}` |
| Calibration method | Sigmoid / Platt scaling |
| Policy type | Capacity-constrained top-k review prioritisation |
| Review capacity | `{int(holdout_result["review_capacity"]):,}` transactions |
| Policy version | `{holdout_result["policy_version"]}` |
| Final evaluation period | Untouched chronological holdout |

The XGBoost model was selected after comparison with Logistic Regression using
identical chronological splits and ranking-focused metrics. Calibration was
selected on chronological validation periods before the final holdout was
evaluated.

## Calibration Evidence

| Method | Brier score | Expected calibration error | PR-AUC | ROC-AUC |
| --- | ---: | ---: | ---: | ---: |
| Uncalibrated | {float(uncalibrated_result["brier_score"]):.6f} | {float(uncalibrated_result["expected_calibration_error"]):.6f} | {float(uncalibrated_result["pr_auc"]):.6f} | {float(uncalibrated_result["roc_auc"]):.6f} |
| Sigmoid / Platt | {float(sigmoid_result["brier_score"]):.6f} | {float(sigmoid_result["expected_calibration_error"]):.6f} | {float(sigmoid_result["pr_auc"]):.6f} | {float(sigmoid_result["roc_auc"]):.6f} |
| Isotonic | {float(isotonic_result["brier_score"]):.6f} | {float(isotonic_result["expected_calibration_error"]):.6f} | {float(isotonic_result["pr_auc"]):.6f} | {float(isotonic_result["roc_auc"]):.6f} |

Sigmoid / Platt scaling was selected because it produced the lowest validation
Brier score and expected calibration error. It improved Brier score from
`{float(uncalibrated_result["brier_score"]):.6f}` to
`{float(sigmoid_result["brier_score"]):.6f}` and ECE from
`{float(uncalibrated_result["expected_calibration_error"]):.6f}` to
`{float(sigmoid_result["expected_calibration_error"]):.6f}`.

## Final Holdout Results

| Metric | Final holdout result |
| --- | ---: |
| Review capacity | {int(holdout_result["review_capacity"]):,} |
| Selected review count | {int(holdout_result["selected_review_count"]):,} |
| Total fraud-labelled transactions | {int(holdout_result["total_fraud_count"]):,} |
| Captured fraud | {int(holdout_result["captured_fraud_count"]):,} |
| Missed fraud | {int(phase7b_policy_summary["missed_fraud_count"]):,} |
| Fraud capture rate | {float(holdout_result["fraud_capture_rate"]):.6f} |
| Legitimate transactions reviewed | {int(holdout_result["false_positive_review_count"]):,} |
| Review precision | {float(phase7b_policy_summary["review_precision"]):.6f} |
| Illustrative review cost | GBP {float(holdout_result["total_expected_review_cost"]):,.2f} |
| Illustrative prevention value | GBP {float(holdout_result["total_expected_prevention_value"]):,.2f} |
| Illustrative net expected value | GBP {float(holdout_result["net_expected_value"]):,.2f} |

The `GBP {float(phase7d_business_value["illustrative_residual_missed_fraud_exposure"]):,.2f}`
residual missed-fraud figure is reported separately as illustrative exposure under
the documented false-negative cost assumption. It is not automatically subtracted
from the saved Phase 5 net-expected-value definition.

## Phase 6 Ablations

Advanced methods were evaluated as controlled ablations using the same temporal
evaluation principles and review-capacity framing. They were not retained as the
champion unless they showed clear measurable decision value.

{ablation_section}

## Explainability and Error Analysis

Global SHAP analysis found `{top_global_features.iloc[0]["transformed_feature"]}`
as the largest mean-absolute contributor within the deterministic final-holdout
explanation sample. SHAP values describe how the fitted model features contributed
to a prediction; they do not prove fraud or establish causal drivers.

The top-10 local contributor sets were stable across the tested deterministic SHAP
background samples, with mean pairwise Jaccard similarity ranging from
`{stability_mean_minimum:.6f}` to `{stability_mean_maximum:.6f}`.

At the fixed review capacity, Phase 7B identified
`{int(phase7b_policy_summary["missed_fraud_count"]):,}` fraud-labelled transactions
outside the review cohort and
`{int(phase7b_policy_summary["unnecessary_review_count"]):,}` legitimate
transactions inside it. Chronological fraud-capture variation confirms that a
single final result should be interpreted with temporal caution.

## Limitations

- The IEEE-CIS dataset is public benchmark data with obfuscated variables,
  missingness, and limited business semantics.
- The analysis uses chronological splits, but benchmark performance cannot
  guarantee performance in other institutions, periods, or fraud environments.
- Calibration, costs, prevention value, and review capacity are documented
  benchmark assumptions rather than real operational policy inputs.
- SHAP provides model attribution only; it does not establish causality, confirm
  fraud, or justify automated adverse action.
- The review policy is capacity constrained. Different capacity, investigation
  quality, customer-friction costs, or intervention effectiveness would change
  the resulting trade-offs.
- No API, streaming, monitoring, cloud deployment, or production service is
  included. This repository is a reproducible analytical portfolio project.

## Portfolio Claim

> This project evaluates fraud prioritisation using leakage-safe chronological
> validation, point-in-time behavioural features, calibrated probabilities,
> explicit cost assumptions and constrained investigation capacity. It compares
> simple and advanced methods through controlled ablations, explains prioritised
> cases, analyses errors and reports expected decision value under documented
> benchmark assumptions.

## Supporting Reports

- `reports/evaluation/explainability_report.md`
- `reports/evaluation/error_analysis_report.md`
- `reports/evaluation/calibration_and_policy_report.md`
- `reports/evaluation/business_value_report.md`
- `reports/tables/phase7e_advanced_ablation_summary.csv`
"""

final_report_path = (
    REPORT_DIRECTORY / "final_model_selection_report.md"
)

final_report_path.write_text(
    final_model_selection_report,
    encoding="utf-8",
)

readme_path = PROJECT_ROOT / "README.md"
existing_readme = readme_path.read_text(encoding="utf-8")

readme_final_results_section = f"""
<!-- FINAL_RESULTS:START -->

## Final Model and Results

### Selected approach

The final benchmark model is **XGBoost** (`{champion_model["model_version"]}`),
with sigmoid / Platt calibration and a constrained top-
`{int(holdout_result["review_capacity"]):,}` transaction review policy.

The model family, feature set, calibration method, and policy were selected using
chronological training and validation periods. The final chronological holdout was
reserved for locked evaluation.

### Final holdout performance

| Metric | Result |
| --- | ---: |
| Review capacity | {int(holdout_result["review_capacity"]):,} |
| Fraud-labelled transactions | {int(holdout_result["total_fraud_count"]):,} |
| Captured fraud | {int(holdout_result["captured_fraud_count"]):,} |
| Fraud capture rate | {float(holdout_result["fraud_capture_rate"]):.2%} |
| Legitimate transactions reviewed | {int(holdout_result["false_positive_review_count"]):,} |
| Review precision | {float(phase7b_policy_summary["review_precision"]):.2%} |
| Illustrative net expected value | GBP {float(holdout_result["net_expected_value"]):,.2f} |

The net expected value is a benchmark calculation under documented illustrative
cost assumptions. It is not realised savings or a real financial-institution
forecast.

### Why this model

- XGBoost was selected after chronological comparison against Logistic Regression.
- Sigmoid calibration reduced validation Brier score from
  `{float(uncalibrated_result["brier_score"]):.4f}` to
  `{float(sigmoid_result["brier_score"]):.4f}` and ECE from
  `{float(uncalibrated_result["expected_calibration_error"]):.4f}` to
  `{float(sigmoid_result["expected_calibration_error"]):.4f}`.
- The final policy respects a fixed investigation capacity instead of using an
  arbitrary probability threshold.
- Phase 6 anomaly, graph, and sequence experiments were treated as controlled
  ablations; the simpler calibrated XGBoost model remained the selected champion.
- Phase 7 includes global and local SHAP attribution, SHAP stability checks,
  missed-fraud and unnecessary-review analysis, and capacity/value trade-offs.

### Main limitations

- This is an IEEE-CIS public-data benchmark, not a production fraud system.
- SHAP contributors describe model behaviour and do not prove fraud or causation.
- Review capacity and financial values are documented illustrative assumptions.
- Results should not be presented as actual fraud savings, institutional
  performance, or a deployed financial-service product.

See the final summary in
[`reports/evaluation/final_model_selection_report.md`](reports/evaluation/final_model_selection_report.md).

<!-- FINAL_RESULTS:END -->
""".strip()

start_marker = "<!-- FINAL_RESULTS:START -->"
end_marker = "<!-- FINAL_RESULTS:END -->"

if start_marker in existing_readme and end_marker in existing_readme:
    section_start = existing_readme.index(start_marker)
    section_end = existing_readme.index(end_marker) + len(end_marker)

    updated_readme = (
        existing_readme[:section_start].rstrip()
        + "\n\n"
        + readme_final_results_section
        + "\n"
        + existing_readme[section_end:].lstrip()
    )
else:
    updated_readme = (
        existing_readme.rstrip()
        + "\n\n"
        + readme_final_results_section
        + "\n"
    )

readme_path.write_text(updated_readme, encoding="utf-8")

print("=== PHASE 7E PORTFOLIO SUMMARY COMPLETE ===")
print(
    "Final report: "
    f"{final_report_path.relative_to(PROJECT_ROOT)}"
)
print(f"Final report size: {final_report_path.stat().st_size:,} bytes")
print("README updated safely with FINAL_RESULTS markers")
print(
    "Selected model: "
    f"{champion_model['model_version']}"
)
print(
    "Final holdout fraud capture rate: "
    f"{float(holdout_result['fraud_capture_rate']):.6f}"
)
print(
    "Final holdout illustrative net expected value: "
    f"GBP {float(holdout_result['net_expected_value']):,.2f}"
)
print(
    "Advanced ablation summaries included: "
    f"{len(ablation_summary)}"
)